# Differential Equations — Session 9
## Section 2.6: Euler's Numerical Method

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. derive Euler's update from a tangent-line approximation.
2. implement Euler's method.
3. construct and interpret an approximation table.
4. calculate absolute and relative error.
5. explain how error changes with step size.
6. recognize that a smaller step can improve accuracy but increase cost.
7. identify a numerical stability failure.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Suggested pacing

| Time | Topic |
|---:|---|
| 0–15 min | Tangent-line derivation |
| 15–32 min | Hand computation |
| 32–50 min | Python implementation and table |
| 50–65 min | Step-size convergence |
| 65–80 min | Numerical stability |
| 80–88 min | Comparison with adaptive solvers |
| 88–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import erf
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=5, suppress=True)

def slope_field(f, xlim=(-3, 3), ylim=(-3, 3), density=21, title=None):
    x = np.linspace(*xlim, density)
    y = np.linspace(*ylim, density)
    X, Y = np.meshgrid(x, y)
    S = np.asarray(f(X, Y), dtype=float)
    S = np.nan_to_num(S, nan=0.0, posinf=20.0, neginf=-20.0)
    U = np.ones_like(S)
    length = np.sqrt(U**2 + S**2)
    plt.quiver(X, Y, U/length, S/length, angles="xy", pivot="mid")
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.xlabel("x")
    plt.ylabel("y")
    if title:
        plt.title(title)

def euler_method(f, x0, y0, h, n_steps):
    xs = np.empty(n_steps + 1)
    ys = np.empty(n_steps + 1)
    xs[0], ys[0] = x0, y0
    for n in range(n_steps):
        ys[n+1] = ys[n] + h*f(xs[n], ys[n])
        xs[n+1] = xs[n] + h
    return xs, ys

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 2.6-A — Euler's method

For the IVP

$$
y'=f(x,y),
\qquad
y(x_0)=y_0,
$$

Euler's method with step size $h$ is

$$
x_{n+1}=x_n+h,
\qquad
y_{n+1}=y_n+h f(x_n,y_n).
$$

### Definition 2.6-B — Errors

At $x_n$:

- Absolute error:
  $$
  E_n=\left|y(x_n)-y_n\right|.
  $$
- Relative error:
  $$
  R_n=\frac{|y(x_n)-y_n|}{|y(x_n)|},
  $$
  provided $y(x_n)\ne0$.
- Percentage relative error: $100R_n\%$.

### Theorem 2.6-C — First-order convergence

Suppose $f$ is sufficiently smooth and Lipschitz continuous in $y$ on a region containing the exact solution. For a fixed final interval, Euler's method has:

- local truncation error of order $O(h^2)$,
- global error of order $O(h)$.

Thus, for sufficiently small $h$, halving the step size approximately halves the global error.

### Stability proposition for the test equation

For

$$
y'=\lambda y,
$$

Euler's method gives

$$
y_{n+1}=(1+h\lambda)y_n.
$$

When $\lambda<0$, numerical decay requires

$$
|1+h\lambda|<1.
$$

### Classroom Checkpoint — One Euler Step

For $y'=x-y$, starting at $(x_0,y_0)=(0,1)$ with $h=0.2$, compute $y_1$.

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Tangent-line update

For

$$
y'=f(x,y),\qquad y(x_0)=y_0,
$$

the tangent line at $(x_n,y_n)$ predicts

$$
y(x_n+h)\approx y_n+h f(x_n,y_n).
$$

Euler's method is therefore

$$
x_{n+1}=x_n+h,
$$

$$
y_{n+1}=y_n+h f(x_n,y_n).
$$

## 2. One Euler path as successive tangent steps

Use

$$
y'=x+y,\qquad y(0)=1.
$$

The exact solution is

$$
y=2e^x-x-1.
$$

In [ ]:
def f(x, y):
    return x+y

def exact(x):
    return 2*np.exp(x)-x-1

h = 0.4
xs, ys = euler_method(f, 0, 1, h, 5)

x_dense = np.linspace(0, 2, 500)
plt.plot(x_dense, exact(x_dense), linewidth=2, label="exact solution")
plt.plot(xs, ys, marker="o", linestyle="--", label="Euler points")

# Draw each tangent segment
for n in range(len(xs)-1):
    segment_x = np.linspace(xs[n], xs[n+1], 20)
    segment_y = ys[n] + f(xs[n], ys[n])*(segment_x-xs[n])
    plt.plot(segment_x, segment_y)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Euler's method follows successive tangent lines")
plt.legend()
plt.show()

## 3. Approximation table

For each point, compare the Euler approximation with the exact value.

In [ ]:
exact_values = exact(xs)
absolute_error = np.abs(exact_values-ys)
relative_error = absolute_error/np.abs(exact_values)

print(f"{'n':>2} {'x_n':>8} {'Euler':>12} {'Exact':>12} {'Abs. error':>12} {'% rel.':>10}")
for n, (x_n, y_n, y_e, err, rel) in enumerate(
    zip(xs, ys, exact_values, absolute_error, relative_error)
):
    print(f"{n:2d} {x_n:8.3f} {y_n:12.6f} {y_e:12.6f} {err:12.6f} {100*rel:10.4f}")

## 4. Step-size experiment

Euler's global error is generally first order:

$$
\text{error}\approx Ch.
$$

Halving $h$ should approximately halve the error when $h$ is sufficiently small.

In [ ]:
target = 2.0
step_sizes = np.array([0.4, 0.2, 0.1, 0.05, 0.025])
errors = []

for h in step_sizes:
    n_steps = int(round(target/h))
    xs_h, ys_h = euler_method(f, 0, 1, h, n_steps)
    errors.append(abs(ys_h[-1]-exact(target)))

errors = np.array(errors)

for h, err in zip(step_sizes, errors):
    print(f"h={h:7.4f}  endpoint error={err:.8f}")

plt.loglog(step_sizes, errors, marker="o", label="observed error")
plt.loglog(step_sizes, errors[0]*(step_sizes/step_sizes[0]),
           linestyle="--", label="reference proportional to h")
plt.xlabel("step size h")
plt.ylabel("endpoint absolute error")
plt.title("First-order convergence of Euler's method")
plt.legend()
plt.show()

## 5. Interactive step-size comparison

In [ ]:
def euler_comparison(h=0.25):
    target = 2.0
    n_steps = int(np.ceil(target/h))
    h_adjusted = target/n_steps
    xs_h, ys_h = euler_method(f, 0, 1, h_adjusted, n_steps)

    x_dense = np.linspace(0, target, 600)
    plt.plot(x_dense, exact(x_dense), linewidth=2, label="exact")
    plt.plot(xs_h, ys_h, marker="o", linestyle="--",
             label=fr"Euler, $h={h_adjusted:.4f}$")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Effect of the step size")
    plt.legend()
    plt.show()

    print("endpoint approximation:", ys_h[-1])
    print("endpoint error:", abs(ys_h[-1]-exact(target)))

if WIDGETS_AVAILABLE:
    interact(
        euler_comparison,
        h=FloatSlider(min=0.025, max=0.5, step=0.025, value=0.25)
    )
else:
    euler_comparison()

## 6. Smaller is not the only issue: numerical stability

Consider

$$
y'=-8y,\qquad y(0)=1.
$$

The exact solution decays:

$$
y=e^{-8t}.
$$

Euler gives

$$
y_{n+1}=(1-8h)y_n.
$$

For numerical decay, we require

$$
|1-8h|<1,
$$

which means

$$
0<h<0.25.
$$

If $h=0.3$, the Euler values alternate and grow even though the exact solution decays.

In [ ]:
def decay_rhs(t, y):
    return -8*y

t_dense = np.linspace(0, 2, 600)
plt.plot(t_dense, np.exp(-8*t_dense), linewidth=2, label="exact")

for h in [0.1, 0.2, 0.3]:
    n_steps = int(np.floor(2/h))
    xs_h, ys_h = euler_method(decay_rhs, 0, 1, h, n_steps)
    plt.plot(xs_h, ys_h, marker="o", linestyle="--", label=fr"Euler $h={h}$")

plt.ylim(-3, 3)
plt.xlabel("t")
plt.ylabel("y")
plt.title("A step size can be numerically unstable")
plt.legend()
plt.show()

## 7. Comparison with an adaptive solver

`solve_ivp` automatically changes its step size to control estimated local error. It is more sophisticated than Euler's method, but Euler remains valuable because it exposes the logic behind numerical integration.

In [ ]:
sol = solve_ivp(lambda x, y: [x+y[0]], (0, 2), [1],
                rtol=1e-8, atol=1e-10, dense_output=True)

x_dense = np.linspace(0, 2, 600)
plt.plot(x_dense, exact(x_dense), label="exact")
plt.plot(x_dense, sol.sol(x_dense)[0], linestyle="--", label="solve_ivp")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

print("Number of internally accepted time points:", len(sol.t))
print("Maximum displayed-grid error:",
      np.max(np.abs(sol.sol(x_dense)[0]-exact(x_dense))))

## Interactive exploration — Accuracy versus computational work

This experiment reports both the endpoint error and the number of Euler steps. It makes the accuracy–cost tradeoff explicit.

In [ ]:
def accuracy_cost(h=0.2):
    def f(x, y):
        return x+y

    exact_value = 2*np.exp(2)-3
    n_steps = int(np.ceil(2/h))
    h_used = 2/n_steps
    xs, ys = euler_method(f, 0, 1, h_used, n_steps)
    error = abs(ys[-1]-exact_value)

    print("Step size used:", h_used)
    print("Number of steps:", n_steps)
    print("Euler approximation:", ys[-1])
    print("Endpoint absolute error:", error)

if WIDGETS_AVAILABLE:
    interact(
        accuracy_cost,
        h=FloatSlider(min=0.01, max=0.5, step=0.01, value=0.2)
    )
else:
    accuracy_cost()

## Classroom Checkpoint — Exit Check

For Euler's method applied to $y'=-5y$, what is the largest $h$ that satisfies the strict stability condition?

> Pause here. Let students commit to an answer before running the next cell.